# Retail dbt Test Generation Pipeline

End-to-end pipeline: PostgreSQL retail data → direct schema extraction from PostgreSQL → Ollama LLM → dbt test generation → dbt test run → results preview.

In [1]:
# Needed to clear the .env variabel path

import os

from dotenv import load_dotenv

os.environ.pop("POST_TEST", None)
os.environ.pop("POSTGRES_HOST", None)

load_dotenv("../.env", override=True)


True

In [2]:
# Cell 1: Setup & Connectivity Check
import os
import psycopg2
import requests

from dotenv import load_dotenv

load_dotenv(os.path.join(os.getcwd(), "..", ".env"))

# Config from env
PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_PORT = int(os.getenv("POSTGRES_PORT", "5435"))
PG_USER = os.getenv("POSTGRES_USER", "retail")
PG_PASSWORD = os.getenv("POSTGRES_PASSWORD", "retail123")
PG_DB = os.getenv("POSTGRES_DB", "retail")
PG_SCHEMA = os.getenv("POSTGRES_SCHEMA", "public")

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "localhost")
OLLAMA_PORT = int(os.getenv("OLLAMA_PORT", "11434"))
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5-coder:3b")

DBT_PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "dbt_project"))

# Connectivity checks
checks = {}

# PostgreSQL
try:
    conn = psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        user=PG_USER,
        password=PG_PASSWORD,
        dbname=PG_DB,
        connect_timeout=5,
    )
    conn.close()
    checks["PostgreSQL"] = True
except Exception as e:
    checks["PostgreSQL"] = str(e)

# Ollama
try:
    r = requests.get(f"http://{OLLAMA_HOST}:{OLLAMA_PORT}/api/tags", timeout=5)
    checks["Ollama"] = r.status_code == 200
except Exception as e:
    checks["Ollama"] = str(e)

print("=== Connectivity Checks ===")
for service, status in checks.items():
    icon = "OK" if status is True else "FAIL"
    print(f"  [{icon}] {service}: {status if status is not True else 'connected'}")


=== Connectivity Checks ===
  [OK] PostgreSQL: connected
  [OK] Ollama: connected


In [3]:
# Cell 2: Load Table Metadata from PostgreSQL
import sys
from pathlib import Path

package_root = Path.cwd().parent
if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))

from pipeline.dbt_generator import list_tables_with_columns


def load_table_metadata():
    conn = psycopg2.connect(
        host=PG_HOST, port=PG_PORT, user=PG_USER, password=PG_PASSWORD, dbname=PG_DB
    )
    try:
        return list_tables_with_columns(conn, schema=PG_SCHEMA)
    finally:
        conn.close()


table_metadata = load_table_metadata()
print(
    f"Loaded metadata for {len(table_metadata)} tables from PostgreSQL schema '{PG_SCHEMA}'."
)


Loaded metadata for 14 tables from PostgreSQL schema 'public'.


In [4]:
# Cell 3: Extract Schema from PostgreSQL
# Build rich table/column metadata for downstream LLM prompting
import psycopg2
import pandas as pd


def _pick(d, *keys, default=None):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default


def _to_bool_nullable(v):
    if isinstance(v, bool):
        return v
    if v is None:
        return None
    return str(v).strip().lower() in {"yes", "true", "t", "1"}


def fetch_information_schema_map(schema_name, table_list):
    if not table_list:
        return {}

    query = """
    WITH cols AS (
        SELECT
            c.table_schema,
            c.table_name,
            c.column_name,
            c.ordinal_position,
            c.is_nullable,
            c.column_default,
            c.data_type,
            c.udt_name,
            c.character_maximum_length,
            c.numeric_precision,
            c.numeric_scale
        FROM information_schema.columns c
        WHERE c.table_schema = %s
          AND c.table_name = ANY(%s)
    ),
    pk AS (
        SELECT kcu.table_schema, kcu.table_name, kcu.column_name
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu
          ON tc.constraint_name = kcu.constraint_name
         AND tc.table_schema = kcu.table_schema
         AND tc.table_name = kcu.table_name
        WHERE tc.constraint_type = 'PRIMARY KEY'
          AND tc.table_schema = %s
          AND tc.table_name = ANY(%s)
    ),
    fk AS (
        SELECT
            kcu.table_schema,
            kcu.table_name,
            kcu.column_name,
            ccu.table_name AS foreign_table,
            ccu.column_name AS foreign_column
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu
          ON tc.constraint_name = kcu.constraint_name
         AND tc.table_schema = kcu.table_schema
         AND tc.table_name = kcu.table_name
        JOIN information_schema.constraint_column_usage ccu
          ON ccu.constraint_name = tc.constraint_name
         AND ccu.table_schema = tc.table_schema
        WHERE tc.constraint_type = 'FOREIGN KEY'
          AND tc.table_schema = %s
          AND tc.table_name = ANY(%s)
    )
    SELECT
        cols.table_name,
        cols.column_name,
        cols.ordinal_position,
        cols.is_nullable,
        cols.column_default,
        cols.data_type,
        cols.udt_name,
        cols.character_maximum_length,
        cols.numeric_precision,
        cols.numeric_scale,
        (pk.column_name IS NOT NULL) AS is_primary_key,
        (fk.column_name IS NOT NULL) AS is_foreign_key,
        fk.foreign_table,
        fk.foreign_column
    FROM cols
    LEFT JOIN pk
      ON pk.table_schema = cols.table_schema
     AND pk.table_name = cols.table_name
     AND pk.column_name = cols.column_name
    LEFT JOIN fk
      ON fk.table_schema = cols.table_schema
     AND fk.table_name = cols.table_name
     AND fk.column_name = cols.column_name
    ORDER BY cols.table_name, cols.ordinal_position;
    """

    info_map = {}
    with psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DB,
        user=PG_USER,
        password=PG_PASSWORD,
        connect_timeout=5,
    ) as conn2:
        with conn2.cursor() as cur:
            cur.execute(
                query,
                (
                    PG_SCHEMA,
                    table_list,
                    PG_SCHEMA,
                    table_list,
                    PG_SCHEMA,
                    table_list,
                ),
            )
            rows = cur.fetchall()

    for row in rows:
        (
            tname,
            cname,
            ordinal_position,
            is_nullable,
            column_default,
            data_type,
            udt_name,
            char_len,
            num_precision,
            num_scale,
            is_pk,
            is_fk,
            foreign_table,
            foreign_column,
        ) = row

        # Better human-readable datatype
        if data_type == "character varying" and char_len:
            dtype = f"varchar({char_len})"
        elif data_type == "character" and char_len:
            dtype = f"char({char_len})"
        elif data_type == "numeric" and num_precision is not None:
            dtype = (
                f"numeric({num_precision},{num_scale})"
                if num_scale is not None
                else f"numeric({num_precision})"
            )
        else:
            dtype = data_type or udt_name or "unknown"

        info_map[(tname, cname)] = {
            "ordinal_position": ordinal_position,
            "nullable": _to_bool_nullable(is_nullable),
            "default": column_default,
            "data_type": dtype,
            "is_primary_key": bool(is_pk),
            "is_foreign_key": bool(is_fk),
            "references": (
                f"{foreign_table}.{foreign_column}"
                if foreign_table and foreign_column
                else None
            ),
            "character_maximum_length": char_len,
            "numeric_precision": num_precision,
            "numeric_scale": num_scale,
        }

    return info_map


if table_metadata:
    table_rows = []
    column_rows = []
    table_schemas = {}

    table_names = [t["name"] for t in table_metadata]
    info_schema_map = fetch_information_schema_map(PG_SCHEMA, table_names)

    for t in table_metadata:
        table_name = t["name"]
        fqn = t.get("fqn", table_name)
        cols = t.get("columns", []) or []

        table_schemas[table_name] = {
            "name": table_name,
            "fqn": fqn,
            "columns": cols,
        }

        data_types = []
        nullable_count = 0
        pk_count = 0
        fk_count = 0

        for idx, c in enumerate(cols, start=1):
            col_name = _pick(c, "name", "column_name", default=f"col_{idx}")
            meta = info_schema_map.get((table_name, col_name), {})

            data_type = meta.get(
                "data_type",
                _pick(
                    c, "data_type", "dataType", "type", "udt_name", default="unknown"
                ),
            )
            nullable = meta.get(
                "nullable",
                _to_bool_nullable(_pick(c, "is_nullable", "nullable", default=None)),
            )
            default_val = meta.get(
                "default", _pick(c, "column_default", "default", default=None)
            )
            is_pk = meta.get(
                "is_primary_key",
                bool(_pick(c, "is_primary_key", "primary_key", "pk", default=False)),
            )
            is_fk = meta.get(
                "is_foreign_key",
                bool(_pick(c, "is_foreign_key", "foreign_key", "fk", default=False)),
            )

            if nullable is True:
                nullable_count += 1
            if is_pk:
                pk_count += 1
            if is_fk:
                fk_count += 1

            data_types.append(str(data_type))

            column_rows.append(
                {
                    "table_name": table_name,
                    "fqn": fqn,
                    "column_name": col_name,
                    "ordinal_position": meta.get(
                        "ordinal_position",
                        _pick(c, "ordinal_position", "position", default=idx),
                    ),
                    "data_type": data_type,
                    "nullable": nullable,
                    "default": default_val,
                    "is_primary_key": is_pk,
                    "is_foreign_key": is_fk,
                    "references": meta.get(
                        "references",
                        _pick(
                            c,
                            "references",
                            "foreign_table",
                            "foreign_column",
                            default=None,
                        ),
                    ),
                    "comment": _pick(c, "comment", "description", default=None),
                    "character_maximum_length": meta.get(
                        "character_maximum_length",
                        _pick(
                            c, "character_maximum_length", "max_length", default=None
                        ),
                    ),
                    "numeric_precision": meta.get(
                        "numeric_precision", _pick(c, "numeric_precision", default=None)
                    ),
                    "numeric_scale": meta.get(
                        "numeric_scale", _pick(c, "numeric_scale", default=None)
                    ),
                }
            )

        table_rows.append(
            {
                "table_name": table_name,
                "fqn": fqn,
                "column_count": len(cols),
                "nullable_columns": nullable_count,
                "primary_key_columns": pk_count,
                "foreign_key_columns": fk_count,
                "distinct_data_types": len(set(data_types)),
                "data_types": ", ".join(sorted(set(data_types))),
            }
        )

    schema_df = (
        pd.DataFrame(table_rows).sort_values("table_name").reset_index(drop=True)
    )
    columns_df = (
        pd.DataFrame(column_rows)
        .sort_values(["table_name", "ordinal_position"])
        .reset_index(drop=True)
    )

    print("Tables extracted from PostgreSQL:")
    display(schema_df)

    print("Column-level schema details:")
    display(columns_df)

    table_columns = {t["name"]: t["columns"] for t in table_metadata}
    table_names = [t["name"] for t in table_metadata]

else:
    print(
        "No tables found. Make sure seed.py ran and the PostgreSQL schema is populated."
    )
    table_columns = {}
    table_names = []
    table_schemas = {}
    schema_df = pd.DataFrame()
    columns_df = pd.DataFrame()


Tables extracted from PostgreSQL:


,table_name,fqn,column_count,nullable_columns,primary_key_columns,foreign_key_columns,distinct_data_types,data_types
0,customers,public.customers,8,2,1,0,6,"boolean, integer, timestamp without time zone,..."
1,employees,public.employees,8,1,1,2,5,"date, integer, numeric(10,2), varchar(100), va..."
2,inventory_snapshots,public.inventory_snapshots,6,0,1,2,2,"date, integer"
3,loyalty_points,public.loyalty_points,6,0,1,1,2,"integer, timestamp without time zone"
4,order_items,public.order_items,6,0,1,2,2,"integer, numeric(10,2)"
5,orders,public.orders,11,1,1,3,5,"integer, numeric(10,2), timestamp without time..."
6,product_categories,public.product_categories,4,2,1,1,3,"integer, text, varchar(100)"
7,product_reviews,public.product_reviews,7,2,1,2,4,"integer, text, timestamp without time zone, va..."
8,products,public.products,9,1,1,2,5,"boolean, integer, numeric(10,2), varchar(200),..."
9,promotions,public.promotions,7,0,1,0,6,"boolean, integer, numeric(10,2), timestamp wit..."


Column-level schema details:


,table_name,fqn,column_name,ordinal_position,data_type,nullable,default,is_primary_key,is_foreign_key,references,comment,character_maximum_length,numeric_precision,numeric_scale
0,customers,public.customers,customer_id,1,integer,False,nextval('customers_customer_id_seq'::regclass),True,False,NaN,None,NaN,32.0,0.0
1,customers,public.customers,first_name,2,varchar(100),False,NaN,False,False,NaN,None,100.0,NaN,NaN
2,customers,public.customers,last_name,3,varchar(100),False,NaN,False,False,NaN,None,100.0,NaN,NaN
3,customers,public.customers,email,4,varchar(100),True,NaN,False,False,NaN,None,100.0,NaN,NaN
4,customers,public.customers,phone,5,varchar(30),True,NaN,False,False,NaN,None,30.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,suppliers,public.suppliers,contact_name,3,varchar(100),True,NaN,False,False,NaN,None,100.0,NaN,NaN
99,suppliers,public.suppliers,email,4,varchar(100),True,NaN,False,False,NaN,None,100.0,NaN,NaN
100,suppliers,public.suppliers,phone,5,varchar(30),True,NaN,False,False,NaN,None,30.0,NaN,NaN
101,suppliers,public.suppliers,country,6,varchar(100),False,NaN,False,False,NaN,None,100.0,NaN,NaN


In [5]:
# # Cell 4: Sample Data Preview
# from pipeline.dbt_generator import get_sample_rows

# conn = psycopg2.connect(
#     host=PG_HOST, port=PG_PORT, user=PG_USER, password=PG_PASSWORD, dbname=PG_DB
# )

# print("Sample data (5 rows per table):")
# for table in table_names[:5]:  # preview first 5 tables
#     rows = get_sample_rows(conn, table, limit=5)
#     if rows:
#         print(f"\n--- {table} ---")
#         display(pd.DataFrame(rows))


In [6]:
import psycopg2

# Cell 5: Profile All Columns (dbt profiler-style metrics + dbt_expectations hints)
from psycopg2 import sql

conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT, user=PG_USER, password=PG_PASSWORD, dbname=PG_DB
)


In [7]:
import os
import json
import re
import pandas as pd
from pipeline.llm_client import OllamaClient
from pipeline.dbt_generator import get_sample_rows

# Generate column expectation SQL files and profiler metrics using Ollama LLM
EXPECT_DIR = os.path.join(DBT_PROJECT_DIR, "expectations")
PROFILE_DIR = os.path.join(EXPECT_DIR, "profiles")
os.makedirs(EXPECT_DIR, exist_ok=True)
os.makedirs(PROFILE_DIR, exist_ok=True)

ollama = OllamaClient(OLLAMA_HOST, OLLAMA_PORT, OLLAMA_MODEL)
if not ollama.health():
    raise RuntimeError(f"Ollama not reachable at {OLLAMA_HOST}:{OLLAMA_PORT}")


def _slug(s: str) -> str:
    return re.sub(r"[^0-9a-zA-Z]+", "_", str(s)).strip("_").lower()


def _jinja_ref(table: str) -> str:
    return "{{ ref('stg_" + table + "') }}"


def _safe_identifier(name: str) -> str:
    return '"' + str(name).replace('"', '""') + '"'


def _col_sample_values(table: str, col: str, limit: int = 10):
    try:
        rows = get_sample_rows(conn, table, limit=limit)
    except Exception:
        rows = []
    vals = []
    for r in rows:
        if isinstance(r, dict) and col in r:
            v = r[col]
            vals.append(None if v is None else str(v))
    return vals[:limit]


def _get_stats_for(table: str, col: str):
    s = {}

    # Primary source: all_stats[table][col] from profiling cell
    try:
        if isinstance(all_stats, dict):
            s0 = all_stats.get(table, {}).get(col, {})
            if isinstance(s0, dict) and s0:
                return {k: (None if v is None else v) for k, v in s0.items()}
    except Exception:
        pass

    # Fallback source: stats_df lookup
    try:
        row = stats_df[(stats_df["table"] == table) & (stats_df["column"] == col)]
        if not row.empty:
            r = row.iloc[0].to_dict()
            s = {k: (None if pd.isna(v) else v) for k, v in r.items()}
    except Exception:
        s = {}

    return s


PROMPT_TEMPLATE = """
You are a dbt data-quality assistant. Use ONLY the provided schema metadata, column stats, and sample values.
Infer likely column semantics and generate dbt-compatible checks.

Return valid JSON with exactly these keys:
- expectations: array of objects with keys [name, condition, description]
- profiling_sql: SQL string to profile this column on the staging model
- rationale: short text

,
Rules:
- Conditions must reference the column exactly as {col_quoted}.
- Keep SQL ANSI/Postgres compatible.
- Prefer robust checks: not_null, unique, accepted_values, length bounds, regex/email/phone patterns, numeric ranges, date sanity, fk-presence hints.
- Do NOT invent impossible business rules; be conservative when confidence is low.
- If uncertain, emit fewer checks with clear names.

Context:
table={table}
column={col}
data_type={dtype}
nullable={nullable}
is_primary_key={is_primary_key}
is_foreign_key={is_foreign_key}
references={references}
table_columns={table_columns_json}
column_stats={stats_json}
sample_values={samples}

profiling_sql must query from: {relation}
"""

for table in table_names[:2]:
    print(f"\n=== Processing table: {table} ===")
    table_cols = columns_df[columns_df["table_name"] == table].sort_values(
        "ordinal_position"
    )
    if table_cols.empty:
        continue

    table_column_context = [
        {
            "name": r["column_name"],
            "data_type": r.get("data_type"),
            "nullable": bool(r.get("nullable", True)),
            "is_primary_key": bool(r.get("is_primary_key", False)),
            "is_foreign_key": bool(r.get("is_foreign_key", False)),
            "references": r.get("references"),
        }
        for _, r in table_cols.iterrows()
    ]

    for _, crow in table_cols.iterrows():
        print(f"\n--- Column: {crow['column_name']} ---")
        col = crow["column_name"]
        dtype = str(crow.get("data_type", "") or "")
        nullable = bool(crow.get("nullable", True))
        is_primary_key = bool(crow.get("is_primary_key", False))
        is_foreign_key = bool(crow.get("is_foreign_key", False))
        references = crow.get("references")

        relation = _jinja_ref(table)
        col_quoted = _safe_identifier(col)
        stats = _get_stats_for(table, col)
        stats_json = json.dumps(stats, default=str)
        samples = _col_sample_values(table, col, limit=10)

        prompt = PROMPT_TEMPLATE.format(
            table=table,
            col=col,
            col_quoted=col_quoted,
            dtype=dtype,
            nullable=nullable,
            is_primary_key=is_primary_key,
            is_foreign_key=is_foreign_key,
            references=references,
            table_columns_json=json.dumps(table_column_context, default=str),
            stats_json=stats_json,
            samples=json.dumps(samples, default=str),
            relation=relation,
        )

        raw = ollama.generate(prompt)
        print(f"LLM raw response:\n{raw}")
        parsed = None
        try:
            parsed = json.loads(raw)
        except Exception:
            m = re.search(r"(\{.*\})", raw, re.S)
            if m:
                try:
                    parsed = json.loads(m.group(1))
                except Exception:
                    parsed = None

        base_name = f"expect_{_slug(table)}__{_slug(col)}"
        meta_path = os.path.join(PROFILE_DIR, f"{base_name}_meta.json")

        if parsed and isinstance(parsed, dict):
            exps = parsed.get("expectations", []) or []
            profiling_sql = str(parsed.get("profiling_sql", "") or "").strip()

            prof_sql_path = os.path.join(EXPECT_DIR, f"{base_name}__profiling.sql")
            with open(prof_sql_path, "w", encoding="utf-8") as f:
                f.write(f"-- Profiling SQL for {table}.{col}\n")
                f.write(
                    profiling_sql or f"SELECT COUNT(*) AS total_count FROM {relation};"
                )

            for idx, e in enumerate(exps, start=1):
                if not isinstance(e, dict):
                    continue
                name = e.get("name") or f"exp_{idx}"
                cond = str(e.get("condition") or "").strip()
                desc = str(e.get("description") or "").strip()
                if not cond:
                    continue

                test_sql = (
                    f"-- {name}: {desc}\n"
                    f"select * from {relation}\n"
                    f"where not ({cond})\n"
                    "limit 100;\n"
                )
                fname = os.path.join(
                    EXPECT_DIR, f"{base_name}__{idx:02d}_{_slug(name)}.sql"
                )
                with open(fname, "w", encoding="utf-8") as tf:
                    tf.write(test_sql)

            with open(meta_path, "w", encoding="utf-8") as mf:
                json.dump(
                    {
                        "table": table,
                        "column": col,
                        "dtype": dtype,
                        "nullable": nullable,
                        "is_primary_key": is_primary_key,
                        "is_foreign_key": is_foreign_key,
                        "references": references,
                        "samples": samples,
                        "stats": stats,
                        "llm_rationale": parsed.get("rationale"),
                    },
                    mf,
                    default=str,
                    indent=2,
                )
            print(
                f"Generated {len(exps)} expectations for {table}.{col} and saved to {EXPECT_DIR}."
            )

        else:
            raw_path = os.path.join(EXPECT_DIR, f"{base_name}__raw.txt")
            with open(raw_path, "w", encoding="utf-8") as rf:
                rf.write(raw)

            null_rate = stats.get("null_rate", 0) if isinstance(stats, dict) else 0
            try:
                null_rate_val = float(str(null_rate).replace("%", ""))
            except Exception:
                null_rate_val = 0.0

            if (null_rate_val == 0.0) and (not nullable):
                test_sql = (
                    f"-- fallback not_null for {table}.{col}\n"
                    f"select * from {relation}\n"
                    f"where {_safe_identifier(col)} is null\n"
                    "limit 100;\n"
                )
                fname = os.path.join(EXPECT_DIR, f"{base_name}__fallback_not_null.sql")
                with open(fname, "w", encoding="utf-8") as tf:
                    tf.write(test_sql)

            with open(meta_path, "w", encoding="utf-8") as mf:
                json.dump(
                    {
                        "table": table,
                        "column": col,
                        "dtype": dtype,
                        "nullable": nullable,
                        "is_primary_key": is_primary_key,
                        "is_foreign_key": is_foreign_key,
                        "references": references,
                        "samples": samples,
                        "stats": stats,
                        "llm_raw": raw,
                    },
                    mf,
                    default=str,
                    indent=2,
                )


print(f"Saved expectation SQL files to: {EXPECT_DIR}")
print(f"Saved profiler metadata to: {PROFILE_DIR}")



=== Processing table: customers ===

--- Column: customer_id ---
LLM raw response:
```json
[
  {
    "name": "customer_id",
    "condition": "customer_id IS NOT NULL AND customer_id = ANY(@sample_values)",
    "description": "Check if the 'customer_id' column is not null and matches any of the sample values."
  },
  {
    "name": "first_name",
    "condition": "first_name IS NOT NULL AND first_name = ANY(@sample_values)",
    "description": "Check if the 'first_name' column is not null and matches any of the sample values."
  },
  {
    "name": "last_name",
    "condition": "last_name IS NOT NULL AND last_name = ANY(@sample_values)",
    "description": "Check if the 'last_name' column is not null and matches any of the sample values."
  },
  {
    "name": "email",
    "condition": "email IS NOT NULL AND email = ANY(@sample_values)",
    "description": "Check if the 'email' column is not null and matches any of the sample values."
  },
  {
    "name": "phone",
    "condition": "phone I

KeyboardInterrupt: 

In [ ]:
# import psycopg2

# # Cell 5: Profile All Columns (dbt profiler-style metrics + dbt_expectations hints)
# from psycopg2 import sql

# conn = psycopg2.connect(
#     host=PG_HOST, port=PG_PORT, user=PG_USER, password=PG_PASSWORD, dbname=PG_DB
# )


# def _is_numeric(dtype: str) -> bool:
#     dtype = (dtype or "").lower()
#     numeric_tokens = [
#         "int",
#         "numeric",
#         "decimal",
#         "float",
#         "double",
#         "real",
#         "serial",
#         "money",
#     ]
#     return any(tok in dtype for tok in numeric_tokens)


# def _is_bool(dtype: str) -> bool:
#     return "bool" in (dtype or "").lower()


# def profile_column(conn, table_name: str, column_name: str, data_type: str) -> dict:
#     is_numeric = _is_numeric(data_type)
#     is_bool = _is_bool(data_type)

#     numeric_expr = sql.SQL(
#         """
#         COUNT(*) FILTER (WHERE {col} = 0)::bigint AS zero_count,
#         MIN({col})::text AS min_val,
#         MAX({col})::text AS max_val,
#         AVG({col}::numeric) AS avg_numeric,
#         STDDEV_POP({col}::numeric) AS stddev_numeric
#         """
#     ).format(col=sql.Identifier(column_name))

#     non_numeric_expr = sql.SQL(
#         """
#         NULL::bigint AS zero_count,
#         MIN({col}::text) AS min_val,
#         MAX({col}::text) AS max_val,
#         NULL::numeric AS avg_numeric,
#         NULL::numeric AS stddev_numeric
#         """
#     ).format(col=sql.Identifier(column_name))

#     bool_expr = sql.SQL(
#         """
#         COUNT(*) FILTER (WHERE {col} IS TRUE)::bigint AS true_count,
#         COUNT(*) FILTER (WHERE {col} IS FALSE)::bigint AS false_count
#         """
#     ).format(col=sql.Identifier(column_name))

#     non_bool_expr = sql.SQL("NULL::bigint AS true_count, NULL::bigint AS false_count")

#     query = sql.SQL(
#         """
#         SELECT
#             COUNT(*)::bigint AS total_count,
#             COUNT({col})::bigint AS non_null_count,
#             (COUNT(*) - COUNT({col}))::bigint AS null_count,
#             ROUND((COUNT(*) - COUNT({col}))::numeric / NULLIF(COUNT(*), 0), 6) AS null_rate,
#             COUNT(DISTINCT {col}::text) FILTER (WHERE {col} IS NOT NULL)::bigint AS distinct_count,
#             ROUND(
#                 COUNT(DISTINCT {col}::text) FILTER (WHERE {col} IS NOT NULL)::numeric
#                 / NULLIF(COUNT({col}), 0),
#                 6
#             ) AS uniqueness_score,
#             COUNT(*) FILTER (WHERE NULLIF(BTRIM({col}::text), '') IS NULL AND {col} IS NOT NULL)::bigint AS blank_count,
#             ROUND(AVG(CHAR_LENGTH({col}::text)) FILTER (WHERE {col} IS NOT NULL), 4) AS avg_length,
#             MIN(CHAR_LENGTH({col}::text)) FILTER (WHERE {col} IS NOT NULL) AS min_length,
#             MAX(CHAR_LENGTH({col}::text)) FILTER (WHERE {col} IS NOT NULL) AS max_length,
#             {numeric_or_not},
#             {bool_or_not}
#         FROM {tbl}
#         """
#     ).format(
#         col=sql.Identifier(column_name),
#         tbl=sql.Identifier(table_name),
#         numeric_or_not=numeric_expr if is_numeric else non_numeric_expr,
#         bool_or_not=bool_expr if is_bool else non_bool_expr,
#     )

#     with conn.cursor() as cur:
#         cur.execute(query)
#         row = cur.fetchone()
#         columns = [d[0] for d in cur.description]

#     result = dict(zip(columns, row))
#     result["dbt_expectations_candidates"] = []
#     if (result.get("null_rate") or 0) == 0:
#         result["dbt_expectations_candidates"].append(
#             "dbt_expectations.expect_column_values_to_not_be_null"
#         )
#     if (result.get("uniqueness_score") or 0) >= 0.99:
#         result["dbt_expectations_candidates"].append(
#             "dbt_expectations.expect_column_values_to_be_unique"
#         )
#     if (result.get("distinct_count") or 0) > 1:
#         result["dbt_expectations_candidates"].append(
#             "dbt_expectations.expect_column_proportion_of_unique_values_to_be_between"
#         )

#     return result


# print("Profiling all columns across all tables...")
# all_stats = {}  # table_name -> col_name -> stats
# profile_rows = []

# for table in table_names:
#     all_stats[table] = {}
#     table_cols_df = columns_df[columns_df["table_name"] == table]
#     for _, col_row in table_cols_df.iterrows():
#         col_name = col_row["column_name"]
#         data_type = str(col_row.get("data_type", "unknown"))
#         stats = profile_column(conn, table, col_name, data_type)
#         all_stats[table][col_name] = stats

#         profile_rows.append(
#             {
#                 "table": table,
#                 "column": col_name,
#                 "data_type": data_type,
#                 "total_count": stats.get("total_count"),
#                 "null_count": stats.get("null_count"),
#                 "null_rate": round(float(stats.get("null_rate") or 0), 6),
#                 "distinct_count": stats.get("distinct_count"),
#                 "uniqueness_score": round(float(stats.get("uniqueness_score") or 0), 6),
#                 "blank_count": stats.get("blank_count"),
#                 "avg_length": stats.get("avg_length"),
#                 "min_length": stats.get("min_length"),
#                 "max_length": stats.get("max_length"),
#                 "zero_count": stats.get("zero_count"),
#                 "min_val": stats.get("min_val"),
#                 "max_val": stats.get("max_val"),
#                 "avg_numeric": stats.get("avg_numeric"),
#                 "stddev_numeric": stats.get("stddev_numeric"),
#                 "true_count": stats.get("true_count"),
#                 "false_count": stats.get("false_count"),
#                 "dbt_expectations_candidates": ", ".join(
#                     stats.get("dbt_expectations_candidates", [])
#                 ),
#             }
#         )
#     print(f"  {table}: {len(table_cols_df)} columns profiled")

# stats_df = (
#     pd.DataFrame(profile_rows)
#     .sort_values(["null_rate", "uniqueness_score"], ascending=[False, False])
#     .reset_index(drop=True)
# )

# print("\nColumn profile summary (all types):")
# display(stats_df.head(50))
# print(f"\nTotal profiled columns: {len(stats_df)}")


Profiling all columns across all tables...
  customers: 8 columns profiled
  employees: 8 columns profiled
  inventory_snapshots: 6 columns profiled
  loyalty_points: 6 columns profiled
  order_items: 6 columns profiled
  orders: 11 columns profiled
  product_categories: 4 columns profiled
  product_reviews: 7 columns profiled
  products: 9 columns profiled
  promotions: 7 columns profiled
  returns: 7 columns profiled
  shipments: 7 columns profiled
  stores: 10 columns profiled
  suppliers: 7 columns profiled

Column profile summary (all types):


,table,column,data_type,total_count,null_count,null_rate,distinct_count,uniqueness_score,blank_count,avg_length,min_length,max_length,zero_count,min_val,max_val,avg_numeric,stddev_numeric,true_count,false_count,dbt_expectations_candidates
0,orders,promotion_id,integer,20000,15951,0.797550,100,0.024697,0,1.9158,1,3,0.0,1,100,50.0750802667325265,29.1332397712646418,NaN,NaN,dbt_expectations.expect_column_proportion_of_u...
1,shipments,delivered_at,timestamp without time zone,18000,13567,0.753722,4433,1.000000,0,25.8813,22,26,NaN,2024-06-21 20:14:16.115963,2026-05-29 20:14:16.190013,None,None,NaN,NaN,dbt_expectations.expect_column_values_to_be_un...
2,shipments,shipped_at,timestamp without time zone,18000,8995,0.499722,9005,1.000000,0,25.8875,22,26,NaN,2024-06-20 20:14:16.065495,2026-05-21 20:14:16.188709,None,None,NaN,NaN,dbt_expectations.expect_column_values_to_be_un...
3,product_categories,parent_category_id,integer,20,5,0.250000,5,0.333333,0,1.0000,1,1,0.0,1,5,2.9333333333333333,1.1234866364235145,NaN,NaN,dbt_expectations.expect_column_proportion_of_u...
4,product_reviews,body,text,8000,500,0.062500,7500,1.000000,0,36.1812,11,70,NaN,Ability fact both game significant pass on.,You when respond high month quickly conference.,None,None,NaN,NaN,dbt_expectations.expect_column_values_to_be_un...
5,employees,manager_id,integer,200,10,0.050000,10,0.052632,0,1.0947,1,2,0.0,1,10,5.6263157894736842,2.8731684518852331,NaN,NaN,dbt_expectations.expect_column_proportion_of_u...
6,customers,email,varchar(100),5000,200,0.040000,4800,1.000000,0,21.7858,15,32,NaN,aallison@example.org,zwebster@example.org,None,None,NaN,NaN,dbt_expectations.expect_column_values_to_be_un...
7,products,cost_price,"numeric(10,2)",500,20,0.040000,477,0.993750,0,5.7208,4,6,0.0,8.08,394.69,167.9912708333333333,93.5863695818466004,NaN,NaN,dbt_expectations.expect_column_values_to_be_un...
8,customers,customer_id,integer,5000,0,0.000000,5000,1.000000,0,3.7786,1,4,0.0,1,5000,2500.5000000000000000,1443.375644106551,NaN,NaN,dbt_expectations.expect_column_values_to_not_b...
9,customers,phone,varchar(30),5000,0,0.000000,5000,1.000000,0,16.1360,10,22,NaN,001-200-214-1884x21630,999-959-9610x02311,None,None,NaN,NaN,dbt_expectations.expect_column_values_to_not_b...



Total profiled columns: 103


In [ ]:
# Cell 6: LLM Test Generation via Ollama
from pipeline.dbt_generator import get_sample_rows
from pipeline.llm_client import OllamaClient, build_prompt, parse_yaml_output

ollama = OllamaClient(OLLAMA_HOST, OLLAMA_PORT, OLLAMA_MODEL)
if not ollama.health():
    print("ERROR: Ollama not reachable at", f"http://{OLLAMA_HOST}:{OLLAMA_PORT}")
else:
    print(f"Ollama ready. Model: {OLLAMA_MODEL}")

per_table_schemas = []

for table in table_names[:2]:
    cols = table_columns.get(table, [])
    stats = all_stats.get(table, {})
    sample_rows = get_sample_rows(conn, table, limit=10)

    prompt = build_prompt(table, cols, stats, sample_rows)

    print(f"Generating tests for {table}...")
    raw = ollama.generate(prompt)

    parsed = parse_yaml_output(raw)
    if parsed:
        models = parsed.get("models", [])
        test_count = sum(len(m.get("columns", [])) for m in models)
        per_table_schemas.append(parsed)
        print(
            f"  Generated schema for {table}: {len(models)} models, ~{test_count} column entries"
        )
    else:
        print(f"  WARNING: Could not parse YAML for {table}. Raw output:")
        print(raw[:200])

print(f"\nGenerated schemas for {len(per_table_schemas)}/{len(table_names)} tables.")


Ollama ready. Model: qwen2.5-coder:0.5b
Generating tests for customers...


KeyboardInterrupt: 

In [ ]:
# Cell 7: Write Generated Files to dbt Project (+ dbt_expectations for all columns)
import yaml
from pipeline.dbt_generator import write_dbt_files, merge_schema_ymls


def _normalized_dbt_type(dtype: str) -> str:
    d = (dtype or "").lower()
    if "int" in d or "serial" in d:
        return "integer"
    if "numeric" in d or "decimal" in d:
        return "numeric"
    if "double" in d or "float" in d or d == "real":
        return "float"
    if "bool" in d:
        return "boolean"
    if "date" in d and "time" not in d:
        return "date"
    if "time" in d:
        return "timestamp"
    if any(tok in d for tok in ["char", "text", "uuid", "json", "jsonb"]):
        return "text"
    return "text"


def _test_name(test_obj):
    if isinstance(test_obj, str):
        return test_obj
    if isinstance(test_obj, dict):
        return next(iter(test_obj.keys()))
    return ""


def add_expectations_for_all_columns(merged_schema_text: str) -> str:
    doc = yaml.safe_load(merged_schema_text) or {"version": 2, "models": []}
    model_map = {m.get("name"): m for m in doc.get("models", [])}

    # Ensure every staging model exists in schema.yml
    for table in table_names:
        model_name = f"stg_{table}"
        if model_name not in model_map:
            model = {
                "name": model_name,
                "description": f"Staging model for {table}",
                "columns": [],
            }
            doc.setdefault("models", []).append(model)
            model_map[model_name] = model

    for table in table_names:
        model_name = f"stg_{table}"
        model = model_map[model_name]
        model.setdefault("columns", [])

        cols_meta = columns_df[columns_df["table_name"] == table].sort_values(
            "ordinal_position"
        )
        col_map = {c.get("name"): c for c in model["columns"] if isinstance(c, dict)}

        # Ensure every discovered column has an entry
        for _, row in cols_meta.iterrows():
            col_name = row["column_name"]
            if col_name not in col_map:
                new_col = {"name": col_name, "tests": []}
                model["columns"].append(new_col)
                col_map[col_name] = new_col

            col_entry = col_map[col_name]
            tests = col_entry.setdefault("tests", [])
            existing = {_test_name(t) for t in tests}

            # Baseline dbt_expectations tests for all columns
            expected_type = _normalized_dbt_type(str(row.get("data_type", "text")))
            if "dbt_expectations.expect_column_values_to_be_of_type" not in existing:
                tests.append(
                    {
                        "dbt_expectations.expect_column_values_to_be_of_type": {
                            "column_type": expected_type
                        }
                    }
                )
                existing.add("dbt_expectations.expect_column_values_to_be_of_type")

            nullable = bool(row.get("nullable", True))
            if (
                not nullable
                and "dbt_expectations.expect_column_values_to_not_be_null"
                not in existing
            ):
                tests.append("dbt_expectations.expect_column_values_to_not_be_null")
                existing.add("dbt_expectations.expect_column_values_to_not_be_null")

            stat = all_stats.get(table, {}).get(col_name, {})
            if (
                (stat.get("uniqueness_score") or 0) >= 0.99
                and "dbt_expectations.expect_column_values_to_be_unique" not in existing
            ):
                tests.append("dbt_expectations.expect_column_values_to_be_unique")
                existing.add("dbt_expectations.expect_column_values_to_be_unique")

    return yaml.dump(doc, default_flow_style=False, sort_keys=False, allow_unicode=True)


merged_yml = merge_schema_ymls(per_table_schemas)
merged_yml = add_expectations_for_all_columns(merged_yml)

print(f"Writing dbt files to: {DBT_PROJECT_DIR}")
write_dbt_files(
    dbt_project_dir=DBT_PROJECT_DIR,
    tables=table_names,
    merged_schema_yml=merged_yml,
    source_name="retail",
)
print("Done writing dbt files with dbt_expectations baseline tests.")


In [ ]:
# Cell 8: Run dbt (deps → run → test → docs generate)
import subprocess


def run_dbt_cmd(args, cwd):
    cmd = ["dbt"] + args + ["--profiles-dir", "."]
    print(f"Running: dbt {' '.join(args)}")
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-3000:])  # last 3000 chars
    if result.stderr:
        print("STDERR:", result.stderr[-1000:])
    return result.returncode


print("=== dbt deps ===")
run_dbt_cmd(["deps"], DBT_PROJECT_DIR)

print("\n=== dbt run (staging) ===")
run_dbt_cmd(["run", "--select", "staging"], DBT_PROJECT_DIR)

print("\n=== dbt test (with store-failures) ===")
run_dbt_cmd(["test", "--store-failures", "--select", "staging"], DBT_PROJECT_DIR)

print("\n=== dbt docs generate ===")
run_dbt_cmd(["docs", "generate"], DBT_PROJECT_DIR)

print("\ndbt commands complete.")


In [ ]:
# Cell 9: Results Dashboard
import json

run_results_path = os.path.join(DBT_PROJECT_DIR, "target", "run_results.json")

if not os.path.exists(run_results_path):
    print("No run_results.json found. Run Cell 8 first.")
else:
    with open(run_results_path) as f:
        run_results = json.load(f)

    results = run_results.get("results", [])
    rows = []
    for r in results:
        node = r.get("unique_id", "")
        node_name = node.split(".")[-1]
        rows.append(
            {
                "test_name": node_name,
                "status": r.get("status", "unknown"),
                "execution_time_s": round(r.get("execution_time", 0), 2),
                "failures": r.get("failures", 0),
                "message": (r.get("message") or "")[:80],
            }
        )

    results_df = pd.DataFrame(rows).sort_values(
        ["status", "failures"], ascending=[True, False]
    )

    total = len(results_df)
    passed = (results_df["status"] == "pass").sum()
    failed = (results_df["status"] == "fail").sum()

    print(f"=== dbt Test Results ===")
    print(f"Total: {total} | Passed: {passed} | Failed: {failed}")
    print()
    display(results_df)


In [ ]:
# Cell 10: Column Metrics Chart
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["figure.figsize"] = (12, 5)

# Chart 1: Test results by status
if "results_df" in dir() and not results_df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    status_counts = results_df["status"].value_counts()
    colors = {
        "pass": "#4CAF50",
        "fail": "#F44336",
        "warn": "#FF9800",
        "error": "#9C27B0",
    }
    bar_colors = [colors.get(s, "#999") for s in status_counts.index]
    ax1.bar(status_counts.index, status_counts.values, color=bar_colors)
    ax1.set_title("dbt Tests by Status")
    ax1.set_xlabel("Status")
    ax1.set_ylabel("Count")
    for i, (idx, val) in enumerate(status_counts.items()):
        ax1.text(i, val + 0.1, str(val), ha="center", va="bottom", fontweight="bold")

    # Chart 2: Top 10 columns by null_rate
    if not stats_df.empty:
        top_nulls = stats_df[stats_df["null_rate"] > 0].nlargest(10, "null_rate")
        if not top_nulls.empty:
            labels = top_nulls["table"] + "." + top_nulls["column"]
            ax2.barh(labels, top_nulls["null_rate"] * 100, color="#F44336", alpha=0.7)
            ax2.set_title("Top 10 Columns by Null Rate")
            ax2.set_xlabel("Null Rate (%)")
            ax2.invert_yaxis()
        else:
            ax2.text(
                0.5,
                0.5,
                "No nulls found!",
                ha="center",
                va="center",
                transform=ax2.transAxes,
            )
            ax2.set_title("Null Rates")

    plt.tight_layout()
    plt.savefig(
        os.path.join("..", "dbt_test_results.png"), dpi=100, bbox_inches="tight"
    )
    plt.show()
    print("Chart saved to dbt_test_results.png")


In [ ]:
# Cell 11: Links & Summary
import subprocess

# Start dbt docs serve in background
docs_proc = subprocess.Popen(
    ["dbt", "docs", "serve", "--port", "8082", "--profiles-dir", "."],
    cwd=DBT_PROJECT_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("=== Pipeline Complete ===")
print(f"Tables discovered from PostgreSQL: {len(table_names)}")
print(f"dbt schemas generated by LLM: {len(per_table_schemas)}")

if "results_df" in dir() and not results_df.empty:
    passed = (results_df["status"] == "pass").sum()
    failed = (results_df["status"] == "fail").sum()
    total = len(results_df)
    print(f"dbt tests: {total} total | {passed} passed | {failed} failed")

print()
print("Links:")
print(f"  dbt Docs:       http://localhost:8082")
print(f"  PostgreSQL:     localhost:{PG_PORT}")
print(f"  Ollama:         http://localhost:{OLLAMA_PORT}")
print()
print("To stop dbt docs server: docs_proc.terminate()")

# Close DB connection
if conn and not conn.closed:
    conn.close()
